# 🎭 Tamil Speech Emotion Recognition (SER) Pipeline
### தமிழ் பேச்சு உணர்ச்சி அறிதல் மாதிரி (Deep Learning on Google Colab)

This notebook trains a **Hybrid 2D CNN + Bidirectional LSTM + Attention** model to recognize emotion in spoken Tamil.

#### 🌟 Key Features:
1. **Bilingual Emotions**: Happy (மகிழ்ச்சி), Sad (சோகம்), Angry (கோபம்), Neutral (இயல்பு), Fear (பயம்), Surprised (ஆச்சரியம்).
2. **Acoustic Signal Processing**: Log-Mel Spectrogram extraction with `torchaudio` & `librosa`.
3. **In-Notebook Audio Player & Spectrogram Visualizer**.
4. **GPU-Accelerated Training** (Colab T4/V100/A100 with AMP).
5. **🎙️ Live Microphone Recording inside Colab** to test predictions in real-time on your own voice!

## 1. 🖥️ Environment & GPU Check

In [ ]:
!nvidia-smi

import torch
import torchaudio
print(f"PyTorch Version: {torch.__version__}")
print(f"Torchaudio Version: {torchaudio.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")

## 2. 📦 Setup Dependencies and Project Code
*(If running directly in Google Colab, uncomment the clone command below)*

In [ ]:
# If cloning repo in Colab:
# !git clone https://github.com/kevinjosh10/ML-MODEL.git
# %cd ML-MODEL

!pip install -q torchaudio librosa soundfile seaborn matplotlib scikit-learn tqdm

## 3. ⚙️ Configuration & Tamil Audio Dataset Preparation
*(Automatically generates starter dataset if no custom audio folder is provided, so you can train immediately)*

In [ ]:
from src.config import Config
from src.data.dataset import get_data_loaders, create_sample_dataset
from src.data.audio_preprocessing import AudioPreprocessor
from src.models import build_model
from src.training.trainer import Trainer
from src.training.metrics import evaluate_model
from src.utils.visualizer import plot_waveform_and_spectrogram, plot_emotion_probabilities
import IPython.display as ipd
import random
import numpy as np

# Initialize configuration
config = Config(
    sample_rate=16000,
    duration=3.0,
    epochs=20,
    batch_size=16,
    learning_rate=1e-3,
    seed=42
)

print("Tamil Emotion Classes:")
for k, v in config.emotion_map.items():
    print(f"  • {k.upper()}: {v}")

train_loader, val_loader, test_loader, classes = get_data_loaders(config)
print(f"\nData split completed: {len(train_loader.dataset)} Train | {len(val_loader.dataset)} Val | {len(test_loader.dataset)} Test")

## 4. 🎧 Audio Player & Spectrogram Visualizer
Listen to sample audio files and inspect their acoustic frequency features.

In [ ]:
preprocessor = AudioPreprocessor(config)
sample_emotion = random.choice(config.classes)
sample_files = list((config.data_dir / sample_emotion).glob("*.wav"))

if sample_files:
    sample_path = sample_files[0]
    print(f"Playing Sample: {sample_path.name} | Emotion: {config.emotion_map[sample_emotion]}")
    
    # Play audio inside notebook
    ipd.display(ipd.Audio(str(sample_path)))
    
    # Visualize Waveform and Mel Spectrogram
    wf = preprocessor.load_audio(sample_path).squeeze(0).cpu().numpy()
    mel = preprocessor.extract_mel_spectrogram(torch.from_numpy(wf).unsqueeze(0)).squeeze(0).cpu().numpy()
    plot_waveform_and_spectrogram(wf, config.sample_rate, mel, title=f"Tamil Voice - {config.emotion_map[sample_emotion]}")

## 5. 🧠 Build & Train the Tamil SER Model
Trains the Hybrid CNN-BiLSTM-Attention network with GPU acceleration.

In [ ]:
model = build_model(config)
print(model)

trainer = Trainer(model, config, train_loader, val_loader)
trainer.fit()

## 6. 📊 Model Evaluation & Confusion Matrix

In [ ]:
best_checkpoint = config.checkpoint_dir / "best_tamil_ser_model.pth"
if best_checkpoint.exists():
    ckpt = torch.load(best_checkpoint)
    model.load_state_dict(ckpt['model_state_dict'])
    
results = evaluate_model(model, test_loader, config)
print(f"\n🎯 Overall Test Accuracy: {results['accuracy']:.2f}%")

## 7. 🎙️ Live Tamil Voice Recording Demo
Click run on the cell below to record your own Tamil speech from your browser microphone!

In [ ]:
from src.utils.audio_recorder import record_audio_in_colab
from src.inference import TamilSERPredictor

# Record 3 seconds of your voice
recorded_file = record_audio_in_colab(filename="my_tamil_speech.wav", duration=3)

if recorded_file:
    ipd.display(ipd.Audio(recorded_file))
    
    predictor = TamilSERPredictor(str(best_checkpoint), config=config)
    prediction = predictor.predict(recorded_file, visualize=True)
    
    print("=" * 50)
    print(f"🎉 Detected Emotion: {prediction['tamil_label']}")
    print(f"✨ Confidence: {prediction['confidence_percentage']}")
    print("=" * 50)